# 1. Prepare MMLU college science and generate teacher material

This notebook downloads the official MMLU test splits for college biology, chemistry, and physics, normalizes them to the existing experiment schema, and generates conditions 0–6 with `deepseek/deepseek-v4-flash-0731` through OpenRouter. Conditions 7 and 8 are deterministic shuffles and do not make API calls.

All generation is resumable. Successful question/condition pairs are skipped when a cell is run again. No cells have been executed in the delivered notebook.

## Setup
Copy `.env.example` to `.env` and set `OPENROUTER_API_KEY`. Install dependencies once with the commented command if needed.

In [1]:
# %pip install -r requirements.txt
from pathlib import Path
import importlib
import sys

EXPERIMENT_ROOT = Path.cwd()
if not (EXPERIMENT_ROOT / 'mmlu_common.py').exists():
    EXPERIMENT_ROOT = (Path.cwd() / 'other_experiments' / 'MMLU data').resolve()
if not (EXPERIMENT_ROOT / 'mmlu_common.py').exists():
    raise FileNotFoundError('Run this notebook from the repository root or its own folder')
sys.path.insert(0, str(EXPERIMENT_ROOT)) if str(EXPERIMENT_ROOT) not in sys.path else None

import mmlu_common
import teacher_generation
importlib.reload(mmlu_common)
importlib.reload(teacher_generation)
print('Experiment root:', EXPERIMENT_ROOT)
print('Teacher model:', teacher_generation.TEACHER_MODEL_ID)

Experiment root: /Users/Farshad/Downloads/analogy codex/other_experiments/MMLU data
Teacher model: deepseek/deepseek-v4-flash-0731


## Download and inspect the questions
This cell downloads dataset files from Hugging Face, but it does not call a language-model API.

In [2]:
dataset_path = teacher_generation.prepare_mmlu_dataset(force=False)
questions = mmlu_common.read_csv(dataset_path)
print('Questions:', len(questions))
print('By subject:')
from collections import Counter
print(Counter(row['mmlu_subject'] for row in questions))
questions[:2]

Questions: 346
By subject:
Counter({'college_biology': 144, 'college_physics': 102, 'college_chemistry': 100})


[{'id': 'mmlu_0000_college_biology',
  'question_stem': 'Based on the characteristic population curves that result from plotting population growth of a species, the most effective means of controlling the mosquito population is to',
  'choices': 'A: maintain the population at a point corresponding to the midpoint of its logistic curve | B: opt for zero population control once the K value of the curve has been reached | C: reduce the carrying capacity cif the environment to lower the K value | D: increase the mortality rate',
  'answer_key': 'C',
  'domain': 'biology',
  'mmlu_subject': 'college_biology',
  'source_index': '0'},
 {'id': 'mmlu_0000_college_chemistry',
  'question_stem': 'The rate, r, of a zero-order chemical reaction A → B can be expressed as which of the following?',
  'choices': 'A: r = k ln[A] | B: r = k [A]^2 | C: r = k [A] | D: r = k',
  'answer_key': 'D',
  'domain': 'chemistry',
  'mmlu_subject': 'college_chemistry',
  'source_index': '0'}]

## Generation settings
`NUM_ROWS` is an expandable per-run boundary. Use a small value for a smoke test, then increase it or set it to `None`. `PROVIDER=None` lets OpenRouter select a compatible provider; set an exact provider ID when provider control is part of the experiment.

In [6]:
TARGET_CONDITIONS = list(range(9))  # 0-6 generated; 7-8 shuffled from condition 0
NUM_ROWS = None                     # None = all college-science questions
START_ROW = 0
CONCURRENCY = 50
TEACHER_MODEL = "deepseek/deepseek-v4-flash-0731"  # Model ID (e.g. openai/gpt-4o, etc.)
PROVIDER = 'deepinfra/fp8'                     # Example: "novita/fp8"
RANDOM_SEED = 42


## Run or resume generation
The next cell makes OpenRouter API calls. It uses DeepSeek V4 Flash for concept extraction and all generated teacher material.

In [7]:
generation_result = teacher_generation.run_generation_pipeline(
    condition_ids=TARGET_CONDITIONS,
    num_rows=NUM_ROWS,
    start_row=START_ROW,
    concurrency=CONCURRENCY,
    teacher_model=TEACHER_MODEL,
    provider=PROVIDER,
    random_seed=RANDOM_SEED,
)
generation_result


Scientific concepts:   0%|          | 0/288 [00:00<?, ?row/s]

Condition 0:   0%|          | 0/326 [00:00<?, ?row/s]

Condition 1:   0%|          | 0/326 [00:00<?, ?row/s]

Condition 2:   0%|          | 0/326 [00:00<?, ?row/s]

Condition 3:   0%|          | 0/326 [00:00<?, ?row/s]

Condition 4:   0%|          | 0/326 [00:00<?, ?row/s]

Condition 5:   0%|          | 0/326 [00:00<?, ?row/s]

Condition 6:   0%|          | 0/326 [00:00<?, ?row/s]

{'dataset': '/Users/Farshad/Downloads/analogy codex/other_experiments/MMLU data/data/mmlu_college_science.csv',
 'selected_rows': 346,
 'conditions': {0: '/Users/Farshad/Downloads/analogy codex/other_experiments/MMLU data/content_conditions/0_MMLU_free-form_300w_unlimited_deepseek-v4-flash_clean.csv',
  1: '/Users/Farshad/Downloads/analogy codex/other_experiments/MMLU data/content_conditions/1_MMLU_free-form_300w_limitedconcept_deepseek-v4-flash_clean.csv',
  2: '/Users/Farshad/Downloads/analogy codex/other_experiments/MMLU data/content_conditions/2_MMLU_free-form_600w_deepseek-v4-flash_clean.csv',
  3: '/Users/Farshad/Downloads/analogy codex/other_experiments/MMLU data/content_conditions/3_MMLU_free-form_2x300w_deepseek-v4-flash_clean.csv',
  4: '/Users/Farshad/Downloads/analogy codex/other_experiments/MMLU data/content_conditions/4_MMLU_free-form_3x200w_deepseek-v4-flash_clean.csv',
  5: '/Users/Farshad/Downloads/analogy codex/other_experiments/MMLU data/content_conditions/5_MMLU_cot

## Validate aligned outputs
Run this after generation. It checks row alignment, answer labels, teacher identity, and the random-control domain constraints.

In [8]:
condition_rows = {}
for condition_id, filename in mmlu_common.CONDITION_FILES.items():
    path = mmlu_common.CONDITION_DIR / filename
    if path.exists():
        condition_rows[condition_id] = mmlu_common.read_csv(path)

expected_label = teacher_generation._default_teacher_label(TEACHER_MODEL)
base_ids = [row["id"] for row in condition_rows[0]]
for condition_id, rows in condition_rows.items():
    assert [row["id"] for row in rows] == base_ids, f"ID mismatch in condition {condition_id}"
    assert all(row["answer_key"] in "ABCD" for row in rows)
    assert all(row["teacher_model"] == expected_label for row in rows)
for row in condition_rows.get(7, []):
    assert row["domain"] == row["analogy_source_domain"] and row["id"] != row["analogy_source_id"]
for row in condition_rows.get(8, []):
    assert row["domain"] != row["analogy_source_domain"]
{condition_id: len(rows) for condition_id, rows in condition_rows.items()}


AssertionError: ID mismatch in condition 2